<a href="https://colab.research.google.com/github/fokkfe/asurada-python/blob/main/primer/day_05_feature_selection_SPY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the data library
!pip install yfinance -q

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

#Download SPY data - 10 years of daily bars
df_raw = yf.download("SPY", start="2015-01-01", end="2025-01-01")

#flatten column names
df_raw.columns = df_raw.columns.get_level_values(0)
df_raw.columns = df_raw.columns.str.lower()

print(df_raw.shape)
print(df_raw.head())

/tmp/ipykernel_28289/2487500934.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_raw = yf.download("SPY", start="2015-01-01", end="2025-01-01")
[*********************100%***********************]  1 of 1 completed

(2516, 5)
Price            close        high         low        open     volume
Date                                                                 
2015-01-02  170.124969  171.325784  169.089793  170.911713  121465900
2015-01-05  167.052567  169.247135  166.746158  169.081509  169632600
2015-01-06  165.479126  167.880730  164.684105  167.358997  209151400
2015-01-07  167.541214  167.880755  166.356978  166.804169  125346700
2015-01-08  170.514252  170.729576  168.932512  168.949065  147217800


One quick thing to note — see the word Price sitting above the column names? That's a leftover from yfinance's multi-level column structure. It's cosmetic, won't affect anything, but if it bothers you later you can remove it with df.columns.name = None.

In [ ]:
#print(df.shape)
#print("---")
#print(df.dtypes)
#print("---")
#print(df.isnull().sum())

*** Create features ***

In [4]:
def create_features(df):
    df = df.copy()   # work on a copy, never touch df_raw

    # Base features
    df['OC']  = (df['close'] - df['open']) / df['open']
    df['HC']  = (df['high'] - df['low']) / df['low']
    df['GAP'] = (df['open'] - df['close'].shift(1)) / df['close'].shift(1)
    df['RET'] = df['close'].pct_change(1)

    # Rolling window features
    for w in [7, 14, 28]:
        df[f'PCHG_{w}'] = df['close'].pct_change(w)
        df[f'VCHG_{w}'] = df['volume'].pct_change(w)
        df[f'MA_{w}']   = df['close'].rolling(w).mean()
        df[f'OC_{w}']   = df['OC'].rolling(w).mean()
        df[f'HC_{w}']   = df['HC'].rolling(w).mean()
        df[f'GAP_{w}']  = df['GAP'].rolling(w).mean()
        df[f'STD_{w}']  = df['close'].rolling(w).std()
        df[f'LB_{w}']   = df[f'MA_{w}'] - 2 * df[f'STD_{w}']
        df[f'UB_{w}']   = df[f'MA_{w}'] + 2 * df[f'STD_{w}']

    # Target
    df['Label'] = df['close'].shift(-1)

    # Drop OHLCV and NaN
    df = df.drop(['open', 'high', 'low', 'close', 'volume'], axis=1)
    df = df.dropna()

    return df

df = create_features(df_raw)
print(df.shape)
print(df.columns.tolist())

(2487, 32)
['OC', 'HC', 'GAP', 'RET', 'PCHG_7', 'VCHG_7', 'MA_7', 'OC_7', 'HC_7', 'GAP_7', 'STD_7', 'LB_7', 'UB_7', 'PCHG_14', 'VCHG_14', 'MA_14', 'OC_14', 'HC_14', 'GAP_14', 'STD_14', 'LB_14', 'UB_14', 'PCHG_28', 'VCHG_28', 'MA_28', 'OC_28', 'HC_28', 'GAP_28', 'STD_28', 'LB_28', 'UB_28', 'Label']


In [5]:
# Separate features from the target label
df1 = df.drop('Label', axis=1)
y = df['Label']

print(df1.shape)

(2487, 31)


In [6]:
corr_matrix = df1.corr().abs()
print(corr_matrix.shape)

(31, 31)


In [7]:
print(corr_matrix.iloc[:5, :5])

Price         OC        HC       GAP       RET    PCHG_7
Price                                                   
OC      1.000000  0.084829  0.034515  0.758157  0.270951
HC      0.084829  1.000000  0.140046  0.152816  0.452920
GAP     0.034515  0.140046  1.000000  0.677773  0.201810
RET     0.758157  0.152816  0.677773  1.000000  0.330025
PCHG_7  0.270951  0.452920  0.201810  0.330025  1.000000


In [8]:
threshold = 0.9

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

print(f"Features dropped: {len(to_drop)}")
print(f"Features kept: {df1.shape[1] - len(to_drop)}")
print(to_drop)

Features dropped: 10
Features kept: 21
['LB_7', 'UB_7', 'MA_14', 'HC_14', 'LB_14', 'UB_14', 'MA_28', 'HC_28', 'LB_28', 'UB_28']


In [9]:
X = df1.drop(to_drop, axis=1)
print(f"Final feature set: {X.shape}")
print(X.columns.tolist())

Final feature set: (2487, 21)
['OC', 'HC', 'GAP', 'RET', 'PCHG_7', 'VCHG_7', 'MA_7', 'OC_7', 'HC_7', 'GAP_7', 'STD_7', 'PCHG_14', 'VCHG_14', 'OC_14', 'GAP_14', 'STD_14', 'PCHG_28', 'VCHG_28', 'OC_28', 'GAP_28', 'STD_28']


In [10]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.0001)
X_selected = selector.fit_transform(X)

print(f"Before variance filter: {X.shape[1]} features")
print(f"After variance filter: {X_selected.shape[1]} features")

Before variance filter: 21 features
After variance filter: 11 features


In [11]:
print(type(X_selected))

<class 'numpy.ndarray'>


In [12]:
# Get the column names that passed the variance filter
selected_mask = selector.get_support()
selected_features = X.columns[selected_mask].tolist()

print(f"Surviving features: {selected_features}")

Surviving features: ['RET', 'PCHG_7', 'VCHG_7', 'MA_7', 'STD_7', 'PCHG_14', 'VCHG_14', 'STD_14', 'PCHG_28', 'VCHG_28', 'STD_28']
